# DEKM_F / MvDEC View1 audit: REUTERS and RCV1, seeds 42-44

This notebook isolates the `DEKM_F` full-batch control described by DEKM 2021 and identified as MvDEC 2025 Proposed View1. It intentionally excludes 20NEWS and does not run View2 or fused MvDEC. The run uses the official DEKM source behavior with the only protocol change being a full-dataset refinement batch.


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
assert gpus, "Enable a GPU accelerator in Kaggle notebook settings."


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path("/kaggle/working/mvdec_fpmax")
subprocess.run(["rm", "-rf", str(REPO)], check=True)
subprocess.run([
    "git", "clone", "--branch", "dev", "--single-branch",
    "https://github.com/vdat-18/mvdec_fpmax.git", str(REPO),
], check=True)
EXPECTED_REVISION = "3a1487d81fc317bfd7f51fcf06a8654b40479c9e"
subprocess.run(
    ["git", "checkout", "--detach", EXPECTED_REVISION], cwd=REPO, check=True
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "gdown"],
    check=True,
)
revision = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO, text=True
).strip()
assert revision == EXPECTED_REVISION, revision
print("Repository revision:", revision)

subprocess.run([
    sys.executable, str(REPO / "scripts/fetch_dekm_release.py"),
    "--datasets", "REUTERS", "RCV1",
], cwd=REPO, check=True)

# The DEKM release stores RCV1 split indices but uses sklearn's source matrix.
from sklearn.datasets import fetch_rcv1
fetch_rcv1(download_if_missing=True)
print("REUTERS and RCV1 sources are ready.")


In [ ]:
import json
import traceback

DATASETS = ("REUTERS", "RCV1")
SEEDS = (42, 43, 44)
PROTOCOL = "dekm_f_2021_official_source_full_batch_v1"
RUN_ROOT = Path("/kaggle/working/dekm-f-reuters-rcv1/runs")
failures = []

for dataset in DATASETS:
    command = [
        sys.executable,
        "-m",
        "representation_learning.DEKM_dense",
        dataset,
        "--protocol",
        "full-batch-2021",
        "--seed",
        str(SEEDS[0]),
        "--runs",
        str(len(SEEDS)),
        "--require-gpu",
        "--dataset-root",
        str(REPO / "external_repos/DEKM/datasets"),
        "--output-dir",
        str(RUN_ROOT),
    ]
    try:
        subprocess.run(command, cwd=REPO, check=True)
    except Exception as error:
        failures.append({
            "dataset": dataset,
            "seeds": list(SEEDS),
            "error_type": type(error).__name__,
            "error": str(error),
        })
        traceback.print_exc()

print("Failures:", json.dumps(failures, indent=2))


In [ ]:
import pandas as pd

PAPER = {
    "REUTERS": {"acc": 0.7642, "nmi": 0.5999},
    "RCV1": {"acc": 0.6212, "nmi": 0.3598},
}
REPORT_ROOT = Path("/kaggle/working/dekm-f-reuters-rcv1")
run_frames = []
manifest_paths = []

for dataset in DATASETS:
    protocol_root = RUN_ROOT / dataset.lower() / PROTOCOL
    config_roots = [path for path in protocol_root.iterdir() if path.is_dir()]
    assert len(config_roots) == 1, (dataset, config_roots)
    config_root = config_roots[0]
    run_frames.append(pd.read_csv(config_root / "dekm_runs.csv"))
    for seed in SEEDS:
        manifest_path = config_root / f"seed_{seed}" / "manifest.json"
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        assert manifest["status"] == "complete"
        assert manifest["method"] == "DEKM_F"
        assert manifest["protocol_id"] == PROTOCOL
        assert manifest["config"]["refinement_batch_size"] == manifest["n_samples"]
        assert manifest["metrics"]["acc"] >= 0
        assert manifest["metrics"]["nmi"] >= 0
        manifest_paths.append(manifest_path)

runs = pd.concat(run_frames, ignore_index=True).sort_values(["dataset", "seed"])
assert len(runs) == len(DATASETS) * len(SEEDS)
assert set(runs["dataset"]) == set(DATASETS)
for dataset in DATASETS:
    assert set(runs.loc[runs["dataset"] == dataset, "seed"]) == set(SEEDS)

aggregate_rows = []
for dataset, group in runs.groupby("dataset", sort=True):
    reference = PAPER[dataset]
    aggregate_rows.append({
        "dataset": dataset,
        "protocol_id": PROTOCOL,
        "completed_runs": len(group),
        "acc_mean": group["acc"].mean(),
        "acc_sample_std": group["acc"].std(ddof=1),
        "nmi_mean": group["nmi"].mean(),
        "nmi_sample_std": group["nmi"].std(ddof=1),
        "paper_acc": reference["acc"],
        "paper_nmi": reference["nmi"],
        "acc_delta_pp": 100 * (group["acc"].mean() - reference["acc"]),
        "nmi_delta_pp": 100 * (group["nmi"].mean() - reference["nmi"]),
        "best_acc": group["acc"].max(),
        "best_nmi": group["nmi"].max(),
        "runtime_seconds_mean": group["fit_time_seconds"].mean(),
        "source_sha256": group["source_sha256"].iloc[0],
        "config_hash": group["config_hash"].iloc[0],
    })

aggregate = pd.DataFrame(aggregate_rows)
runs_path = REPORT_ROOT / "dekm_f_runs.csv"
aggregate_path = REPORT_ROOT / "dekm_f_aggregate.csv"
failures_path = REPORT_ROOT / "failures.json"
runs.to_csv(runs_path, index=False)
aggregate.to_csv(aggregate_path, index=False)
failures_path.write_text(json.dumps(failures, indent=2), encoding="utf-8")

display(runs[[
    "dataset", "seed", "acc", "nmi", "fit_time_seconds", "stop_reason"
]])
display(aggregate)
assert not failures, failures


In [ ]:
import hashlib
import shutil

zip_path = Path(shutil.make_archive(
    "/kaggle/working/dekm-f-reuters-rcv1-seeds42-44",
    "zip",
    REPORT_ROOT,
))
digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()
sha_path = zip_path.with_suffix(".zip.sha256")
sha_path.write_text(f"{digest}  {zip_path.name}\n", encoding="ascii")
print("ZIP:", zip_path)
print("SHA-256:", digest)
print("SHA file:", sha_path)
